## 0. ¿Qué son las APIs?

Una API (Interfaz de Programación de Aplicaciones) es un conjunto de reglas y definiciones que permite que dos aplicaciones de software se comuniquen entre sí. 
En el mundo del software, una API define cómo un programa puede solicitar servicios o datos de otro programa, sin necesidad de conocer los detalles internos de cómo funciona ese otro programa.

## 0.1 ¿Qué es una REST API?

REST (Representational State Transfer) es un estilo de arquitectura para diseñar APIs que funcionan a través de la web. Una API que sigue los principios de REST se llama REST API o RESTful API. Estos son sus principios básicos:

- Cliente-servidor: separa la interfaz de usuario del almacenamiento de datos.

- Sin estado (stateless): cada petición del cliente al servidor debe contener toda la información necesaria para entenderla, sin depender de peticiones anteriores.

- Cacheable: las respuestas deben indicar si son o no almacenables en caché para mejorar el rendimiento.

- Interfaz uniforme: los recursos (datos) se identifican mediante URLs, y se manipulan con métodos HTTP estándar:

    GET → obtener un recurso.

    POST → crear un nuevo recurso.

    PUT / PATCH → actualizar un recurso.

    DELETE → eliminar un recurso.

- Sistema por capas: pueden existir intermediarios (servidores proxy, balanceadores de carga) entre el cliente y el servidor final.


Normalmente, las REST APIs devuelven los datos en formato JSON (JavaScript Object Notation), que es fácil de leer tanto para humanos como para máquinas.

## 1. Introducción a FastAPI

FastAPI es un framework moderno y rápido (de alto rendimiento) para construir APIs con Python 3.7+. Proporciona validación automática de datos, documentación interactiva (Swagger UI) y soporte nativo para operaciones asíncronas.

Al igual que Flask, permite definir rutas y manejar métodos HTTP, pero añade capas de seguridad y validación gracias a Pydantic.

### 1.1. Operaciones CRUD

En una API REST, cada operación CRUD se asocia con un método HTTP:

| Operación | Método HTTP | Ejemplo de uso                  |
|-----------|-------------|----------------------------------|
| Create    | POST        | Crear un nuevo recurso           |
| Read      | GET         | Obtener uno o varios recursos    |
| Update    | PUT / PATCH | Actualizar un recurso existente  |
| Delete    | DELETE      | Eliminar un recurso              |

FastAPI facilita la definición de estos endpoints con decoradores y tipos.

## 2. Instalación de dependencias

Instalamos FastAPI y un servidor ASGI (Asynchronous Server Gateway Interface, un servidor web con capacidades asíncronas) como Uvicorn para ejecutar la aplicación:

In [2]:
%pip install fastapi uvicorn[standard]

  Using cached uvicorn-0.41.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached starlette-0.52.1-py3-none-any.whl.metadata (6.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
Using cached uvicorn-0.41.0-py3-none-any.whl (68 kB)
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 8.0 MB/s  0:00:006m0:00:01
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached python_dotenv-1.2.1-py3-none-any.whl (21 kB)
Using cached starlette-0.52.1-py3-none-any.whl (74 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.9/742.9 kB 9.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [fastapi]0/12 [starlette]pection]
Not

## Creación de la aplicación FastAPI

Importamos las clases necesarias y creamos una instancia de la aplicación.

In [3]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI(title="API de Tareas con FastAPI")

## Definición de modelos Pydantic

Para validar los datos de entrada y salida usaremos modelos de la librería `pydantic`.

In [4]:
class TaskBase(BaseModel):
    titulo: str
    completada: bool = False

class TaskCreate(TaskBase):
    pass

class Task(TaskBase):
    id: int

    class Config:
        from_attributes = True  # para compatibilidad con ORM (opcional)

/var/folders/bt/7bxnp6hd33qdk5yq_vc14kf40000gn/T/ipykernel_57755/1821713272.py:8: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class Task(TaskBase):


## Base de datos en memoria

Simulamos una pequeña colección de tareas.

In [5]:
tasks_db = [
    {"id": 1, "titulo": "Aprender FastAPI", "completada": False},
    {"id": 2, "titulo": "Construir un API CRUD", "completada": False}
]

## GET - Obtener todas las tareas

Endpoint que devuelve la lista completa de tareas.

In [6]:
@app.get("/tasks", response_model=List[Task])
def get_tasks():
    return tasks_db

Probar en el terminal con:
```bash
curl http://127.0.0.1:8000/tasks
```

## POST - Crear una nueva tarea

Recibe los datos en el cuerpo de la petición (JSON) y los valida con el modelo `TaskCreate`.

In [7]:
@app.post("/tasks", response_model=Task, status_code=201)
def create_task(task: TaskCreate):
    new_id = max(t["id"] for t in tasks_db) + 1 if tasks_db else 1
    new_task = task.model_dump()  # convierte el modelo Pydantic a dict
    new_task["id"] = new_id
    tasks_db.append(new_task)
    return new_task

Probar con:
```bash
curl -X POST -H "Content-Type: application/json" -d '{"titulo": "Nueva tarea"}' http://127.0.0.1:8000/tasks
```

## PUT - Actualizar una tarea existente

Actualiza parcial o totalmente una tarea identificada por su `id`.

In [8]:
@app.put("/tasks/{task_id}", response_model=Task)
def update_task(task_id: int, task_update: TaskCreate):
    for t in tasks_db:
        if t["id"] == task_id:
            t["titulo"] = task_update.titulo
            t["completada"] = task_update.completada
            return t
    raise HTTPException(status_code=404, detail="Tarea no encontrada")

Probar con:
```bash
curl -X PUT -H "Content-Type: application/json" -d '{"titulo": "Aprender FastAPI a fondo", "completada": true}' http://127.0.0.1:8000/tasks/1
```

## DELETE - Eliminar una tarea

Borra la tarea con el `id` proporcionado.

In [9]:
@app.delete("/tasks/{task_id}")
def delete_task(task_id: int):
    global tasks_db
    for i, t in enumerate(tasks_db):
        if t["id"] == task_id:
            tasks_db.pop(i)
            return {"result": "Tarea eliminada"}
    raise HTTPException(status_code=404, detail="Tarea no encontrada")

Probar con:
```bash
curl -X DELETE http://127.0.0.1:8000/tasks/2
```

## Ejecutar el servidor

Para lanzar la aplicación, necesitamos un servidor ASGI.
Asegúrate de ejecutar el main.py como un archivo `.py` y luego en un terminal `uvicorn main:app --reload` 

Una vez en marcha, puedes acceder a la documentación interactiva generada automáticamente en:
- Swagger UI: http://127.0.0.1:8000/docs
- ReDoc: http://127.0.0.1:8000/redoc